# Collaborative Network from Git History

Builds two views of contributor collaboration from a GitHub repository:

1. **Network graph** — nodes are contributors, edges come from:
   - *PR collaboration*: both participated (author / reviewer) in the same pull request
   - *Temporal file overlap*: both touched the same code area **and** their activity windows overlap in time
2. **Code Archaeology heatmap** — for each (contributor, code area) pair, measures how many days before the contributor arrived that area was last touched by someone else

## Configuration

In [ ]:
REPO = "torvalds/linux"  # change to any owner/repo
GITHUB_TOKEN = ""         # set a personal access token to raise rate limits
MAX_PRS = 100             # merged PRs to analyse (increase for deeper history)
MAX_COMMIT_PAGES = 5      # x100 commits each, for activity-window computation
MIN_CONTRIBUTIONS = 2     # exclude contributors below this threshold
FILE_GROUP_DEPTH = 1      # directory depth for code-area grouping (1 = top-level dir)

In [ ]:
import time
import requests
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict
from itertools import combinations

## Fetch data from GitHub

In [ ]:
def make_headers(token=''):
    h = {"Accept": "application/vnd.github+json"}
    if token:
        h["Authorization"] = f"Bearer {token}"
    return h


def paginate(url, headers, params=None, max_pages=None):
    results, page = [], 1
    while True:
        p = {**(params or {}), 'page': page, 'per_page': 100}
        r = requests.get(url, headers=headers, params=p, timeout=30)
        if r.status_code == 403:
            raise PermissionError("Rate limit hit — set GITHUB_TOKEN.")
        if r.status_code == 404:
            raise ValueError(f"Not found: {url}")
        r.raise_for_status()
        data = r.json()
        if not data:
            break
        results.extend(data)
        if (max_pages and page >= max_pages) or "next" not in r.links:
            break
        page += 1
    return results

In [ ]:
def fetch_pr_data(repo, headers, max_prs=100):
    print(f"Fetching PRs from {repo}...")
    raw = paginate(
        f"https://api.github.com/repos/{repo}/pulls",
        headers, {"state": "closed"},
        max_pages=(max_prs // 100 + 1) if max_prs else None,
    )
    merged = [pr for pr in raw if pr.get('merged_at')][:max_prs]

    records = []
    for i, pr in enumerate(merged):
        num = pr['number']
        author = (pr.get('user') or {}).get('login')
        date = pd.to_datetime(pr['merged_at'])

        reviews = paginate(
            f"https://api.github.com/repos/{repo}/pulls/{num}/reviews", headers
        )
        reviewers = list(
            {(r.get('user') or {}).get('login') for r in reviews} - {None, author}
        )

        files_data = paginate(
            f"https://api.github.com/repos/{repo}/pulls/{num}/files", headers
        )
        files = [f['filename'] for f in files_data]

        records.append({
            'pr': num, 'author': author,
            'reviewers': reviewers, 'files': files, 'date': date,
        })
        if (i + 1) % 10 == 0:
            print(f"  {i + 1}/{len(merged)} PRs")
        time.sleep(0.05)

    print(f"Done — {len(records)} merged PRs loaded")
    return records


def fetch_commit_activity(repo, headers, max_pages=5):
    print("Fetching commit history for activity windows...")
    commits = paginate(
        f"https://api.github.com/repos/{repo}/commits", headers, max_pages=max_pages
    )
    activity = defaultdict(list)
    for c in commits:
        login = (c.get('author') or {}).get('login')
        date_str = (c.get('commit', {}).get('author') or {}).get('date')
        if login and date_str:
            activity[login].append(pd.to_datetime(date_str))
    print(f"Found activity for {len(activity)} contributors")
    return dict(activity)

## Build collaboration network

In [ ]:
def build_activity_windows(pr_records, commit_activity, min_contributions=2):
    """Merge PR and commit dates into per-contributor (start, end) windows."""
    dates = defaultdict(list)
    for pr in pr_records:
        for person in [pr['author']] + pr['reviewers']:
            if person:
                dates[person].append(pr['date'])
    for person, ds in commit_activity.items():
        dates[person].extend(ds)
    return {
        p: (min(ds), max(ds))
        for p, ds in dates.items()
        if len(ds) >= min_contributions
    }


def overlaps(w1, w2):
    return w1[0] <= w2[1] and w2[0] <= w1[1]


def file_group(path, depth=1):
    parts = path.split('/')
    return '/'.join(parts[:depth]) if len(parts) > depth else parts[0]


def build_network(pr_records, activity_windows, file_group_depth=1):
    G = nx.Graph()
    G.add_nodes_from(activity_windows)

    # PR edges: both participated in the same pull request
    for pr in pr_records:
        participants = list({
            p for p in [pr['author']] + pr['reviewers']
            if p in activity_windows
        })
        for a, b in combinations(participants, 2):
            if G.has_edge(a, b):
                G[a][b]['pr_weight'] = G[a][b].get('pr_weight', 0) + 1
            else:
                G.add_edge(a, b, pr_weight=1, file_weight=0)

    # File + temporal-overlap edges: touched the same code area while both active
    area_contribs = defaultdict(set)
    for pr in pr_records:
        participants = [p for p in [pr['author']] + pr['reviewers'] if p in activity_windows]
        for f in pr['files']:
            area_contribs[file_group(f, file_group_depth)].update(participants)

    for area, people in area_contribs.items():
        for a, b in combinations(people, 2):
            if overlaps(activity_windows[a], activity_windows[b]):
                if G.has_edge(a, b):
                    G[a][b]['file_weight'] = G[a][b].get('file_weight', 0) + 1
                else:
                    G.add_edge(a, b, pr_weight=0, file_weight=1)

    return G

## Network visualisation

In [ ]:
def plot_network(G, repo):
    fig, ax = plt.subplots(figsize=(14, 10))
    pos = nx.spring_layout(G, k=1.5, seed=42)

    pr_only   = [(u, v) for u, v, d in G.edges(data=True) if d.get('pr_weight', 0) > 0 and d.get('file_weight', 0) == 0]
    file_only = [(u, v) for u, v, d in G.edges(data=True) if d.get('pr_weight', 0) == 0 and d.get('file_weight', 0) > 0]
    both      = [(u, v) for u, v, d in G.edges(data=True) if d.get('pr_weight', 0) > 0 and d.get('file_weight', 0) > 0]

    node_sizes = [200 + 80 * G.degree(n) for n in G.nodes()]
    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='steelblue', alpha=0.85, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=pr_only,   edge_color='#e07b39', width=1.5, alpha=0.7, ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=file_only, edge_color='#4caf50', width=1.0, alpha=0.6, ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=both,      edge_color='#9c27b0', width=2.5, alpha=0.8, ax=ax)

    legend = [
        mpatches.Patch(color='#e07b39', label='PR collaboration only'),
        mpatches.Patch(color='#4caf50', label='Shared code area (temporal overlap) only'),
        mpatches.Patch(color='#9c27b0', label='Both'),
    ]
    ax.legend(handles=legend, loc='upper left', fontsize=9)
    ax.set_title(f'Collaboration Network — {repo}', fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

## Code Archaeology

For each *(contributor, code area)* pair, the **archaeology depth** is the number of days between when a contributor first became active and when that code area was last touched by someone else before them.

- **Large value** → contributor arrived long after others left; understanding the code requires digging deep into history
- **Zero / NaN** → contributor was contemporary with others who touched that area (no archaeology needed)

In [ ]:
def build_archaeology_matrix(pr_records, activity_windows, file_group_depth=1,
                              top_files=30, top_contributors=25):
    area_timeline = defaultdict(list)
    for pr in pr_records:
        participants = [p for p in [pr['author']] + pr['reviewers'] if p in activity_windows]
        for f in pr['files']:
            g = file_group(f, file_group_depth)
            for p in participants:
                area_timeline[g].append((pr['date'], p))
    for g in area_timeline:
        area_timeline[g].sort()

    contrib_count = defaultdict(int)
    for pr in pr_records:
        for p in [pr['author']] + pr['reviewers']:
            if p in activity_windows:
                contrib_count[p] += 1
    top_contribs = sorted(contrib_count, key=contrib_count.get, reverse=True)[:top_contributors]

    top_areas = sorted(
        area_timeline, key=lambda g: len(area_timeline[g]), reverse=True
    )[:top_files]

    matrix = pd.DataFrame(index=top_areas, columns=top_contribs, dtype=float)

    for area in top_areas:
        for c in top_contribs:
            c_start = activity_windows[c][0]
            prior = [
                date for date, person in area_timeline[area]
                if person != c and date < c_start
            ]
            if prior:
                matrix.loc[area, c] = float(max((c_start - max(prior)).days, 0))

    return matrix


def plot_archaeology(matrix, repo):
    data = matrix.dropna(how='all').dropna(axis=1, how='all')
    if data.empty:
        print('No archaeology data: all contributors were fully contemporaneous.')
        return

    h = max(6, len(data.index) * 0.35)
    w = max(10, len(data.columns) * 0.55)
    fig, ax = plt.subplots(figsize=(w, h))
    sns.heatmap(
        data.astype(float),
        cmap='YlOrRd',
        ax=ax,
        linewidths=0.3,
        cbar_kws={'label': 'Archaeology depth (days)'},
        mask=data.isna(),
    )
    ax.set_title(
        f'Code Archaeology — {repo}\n'
        'Days since others last touched this code area before a contributor arrived',
        fontsize=11,
    )
    ax.set_xlabel('Contributor')
    ax.set_ylabel('Code Area')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(fontsize=7)
    plt.tight_layout()
    plt.show()

## Run

In [ ]:
HEADERS = make_headers(GITHUB_TOKEN)

pr_records      = fetch_pr_data(REPO, HEADERS, max_prs=MAX_PRS)
commit_activity = fetch_commit_activity(REPO, HEADERS, max_pages=MAX_COMMIT_PAGES)

activity_windows = build_activity_windows(pr_records, commit_activity,
                                           min_contributions=MIN_CONTRIBUTIONS)
print(f'Active contributors (>={MIN_CONTRIBUTIONS} contributions): {len(activity_windows)}')

G = build_network(pr_records, activity_windows, file_group_depth=FILE_GROUP_DEPTH)
print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

plt.style.use('fivethirtyeight')

plot_network(G, REPO)

arch_matrix = build_archaeology_matrix(
    pr_records, activity_windows,
    file_group_depth=FILE_GROUP_DEPTH,
)
plot_archaeology(arch_matrix, REPO)